# Session 4 — FeatureBuilder

Cleaning gave me a gap-free daily AQI series per city. Now I turn that raw series
into a model-ready table. The job of this session is to build features that
describe everything known up to and including day T, paired with a target that is
day T+1's AQI — so the model learns to forecast tomorrow from today and the recent
past, never from tomorrow itself.

## 0. Load the clean frame and lock the contract

Before I build a single feature I reload the cleaned city-day frame, re-sort it by
(city_id, date), and assert the dates climb in order inside every city. Every lag
and rolling feature I build next leans entirely on this ordering — if it's wrong,
the leakage guarantees silently break, with no error to warn me. So I prove it
once, here, with a hard assert.

In [2]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# make the src package importable from inside notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src import config

# --- load the cleaned city-day frame ---
df = pd.read_parquet(config.CLEAN_CITY_DAY_PATH)

# --- re-sort defensively: features depend entirely on this order ---
df = df.sort_values([config.CITY_ID_COL, config.DATE_COL]).reset_index(drop=True)

# --- prove dates climb in order inside EVERY city (the leakage guard) ---
dates_ok = df.groupby(config.CITY_ID_COL)[config.DATE_COL].apply(
    lambda s: s.is_monotonic_increasing
).all()
assert dates_ok, "dates are not strictly ordered within every city"

print("shape       :", df.shape)
print("cities      :", df[config.CITY_ID_COL].nunique())
print("date range  :", df[config.DATE_COL].min(), "->", df[config.DATE_COL].max())
print("columns     :", list(df.columns))
df.head()

shape       : (224808, 15)
cities      : 141
date range  : 2018-01-01 00:00:00 -> 2023-03-31 00:00:00
columns     : ['city_id', 'city', 'state', 'date', 'pm25', 'pm10', 'no', 'no2', 'nox', 'nh3', 'so2', 'co', 'o3', 'aqi', 'aqi_bucket']


,city_id,city,state,date,pm25,pm10,no,no2,nox,nh3,so2,co,o3,aqi,aqi_bucket
0,"Agartala, Tripura",Agartala,Tripura,2020-11-08,53.016667,68.795000,1.323333,6.743333,8.565000,4.320000,11.036667,0.651667,1.64000,88.200575,Satisfactory
1,"Agartala, Tripura",Agartala,Tripura,2020-11-09,42.771304,55.189130,1.668750,6.787083,9.105000,4.706667,11.220833,0.858750,6.51875,70.889445,Satisfactory
2,"Agartala, Tripura",Agartala,Tripura,2020-11-10,41.183750,54.986250,4.335000,8.927917,15.153333,7.411667,11.320000,1.085000,5.01750,68.207026,Satisfactory
3,"Agartala, Tripura",Agartala,Tripura,2020-11-11,49.401667,64.949091,1.315833,7.173333,8.950417,4.712083,11.314167,1.108750,6.50875,82.092471,Satisfactory
4,"Agartala, Tripura",Agartala,Tripura,2020-11-12,45.761250,62.839167,1.043333,7.678333,9.040417,4.743333,11.291667,0.827500,11.22000,75.941422,Satisfactory


## 1. Build the target: tomorrow's AQI

I build the target before any feature, deliberately. Once target_aqi sits in the
frame, the leakage contract becomes visible: this is the only column allowed to
describe day T+1, while every feature I add later must describe day T or earlier.

To get tomorrow's value onto today's row I slide each city's AQI column up by one
with shift(-1). I group by city_id first so the slide never crosses a city
boundary — Delhi's tomorrow must come from Delhi, not from the next city in the
table. This is only correct because cleaning reindexed every city to a gap-free
daily grid, so the next row genuinely is the next calendar day.

The last day of every city has no tomorrow, so its target is NaN by design. I
leave those rows in for now and drop them in the assembly step, once all features
exist and I can drop every unusable row in one honest pass.

In [3]:
# --- target = NEXT day's AQI value and category, per city ---
# shift(-1) slides each city's column UP one row, so tomorrow lands on today.
g = df.groupby(config.CITY_ID_COL)

df["target_aqi"]    = g[config.AQI_COL].shift(-1)
df["target_bucket"] = g[config.AQI_BUCKET_COL].shift(-1)

# --- prove the slide is correct on one city ---
sample = df[df[config.CITY_ID_COL] == df[config.CITY_ID_COL].iloc[0]]
print("checking city:", sample[config.CITY_ID_COL].iloc[0])
print(
    sample[[config.DATE_COL, config.AQI_COL, "target_aqi", "target_bucket"]]
    .head()
    .to_string(index=False)
)

# today's target_aqi must equal tomorrow's aqi (row i target == row i+1 aqi)
today_target = sample["target_aqi"].iloc[0]
tomorrow_aqi = sample[config.AQI_COL].iloc[1]
print("\nrow0 target_aqi :", today_target)
print("row1 aqi        :", tomorrow_aqi)
assert today_target == tomorrow_aqi, "shift(-1) did not align tomorrow onto today"

# every city's LAST row must have a NaN target (no tomorrow exists)
last_rows = df.groupby(config.CITY_ID_COL).tail(1)
assert last_rows["target_aqi"].isna().all(), "some city's last day has a non-NaN target"
print("\nlast-day targets all NaN:", last_rows['target_aqi'].isna().all())
print("total NaN targets       :", df['target_aqi'].isna().sum(), "(expect >= 141)")

checking city: Agartala, Tripura
      date       aqi  target_aqi target_bucket
2020-11-08 88.200575   70.889445  Satisfactory
2020-11-09 70.889445   68.207026  Satisfactory
2020-11-10 68.207026   82.092471  Satisfactory
2020-11-11 82.092471   75.941422  Satisfactory
2020-11-12 75.941422   58.662184  Satisfactory

row0 target_aqi : 70.88944527736132
row1 aqi        : 70.88944527736132

last-day targets all NaN: True
total NaN targets       : 9973 (expect >= 141)


## 2. Lag features: AQI from the recent past

Lag features carry yesterday's answer forward as a clue for today. Air quality is
sticky day to day (it's strongly autocorrelated), so the recent AQI trajectory is
the most powerful signal I have for forecasting tomorrow.

I build four lags: 1, 2, and 3 days back to capture short-term momentum, and 7 days
back to compare the same weekday a week ago, which respects the weekly weekday vs
weekend rhythm. The mechanic mirrors the target: the target used shift(-1) to pull
the future down; lags use shift(+k) to pull the past up. I group by city_id so no
city borrows another's history, and this is only meaningful because the grid is
gap-free — lag_7 genuinely means seven calendar days ago, not seven rows ago.

I lag AQI only, not the pollutants: today's pollutants already sit in the frame as
legal day-T features, while AQI gets the lag treatment because it's the target's
own history. Lagging pollutants is a future lever if models underperform.

In [4]:
# LAG_DAYS promoted to config.py when FeatureBuilder is assembled
LAG_DAYS = [1, 2, 3, 7]

# rebuild the grouping fresh so this cell is self-contained on re-run
g = df.groupby(config.CITY_ID_COL)

# shift(+k) slides each city's AQI column DOWN k rows: the value from k days
# ago lands on today's row.
for k in LAG_DAYS:
    df[f"aqi_lag_{k}"] = g[config.AQI_COL].shift(k)

lag_cols = [f"aqi_lag_{k}" for k in LAG_DAYS]

# --- prove the lag is correct on one city ---
sample = df[df[config.CITY_ID_COL] == df[config.CITY_ID_COL].iloc[0]]
print("checking city:", sample[config.CITY_ID_COL].iloc[0])
print(
    sample[[config.DATE_COL, config.AQI_COL] + lag_cols]
    .head(9)
    .to_string(index=False)
)

# row i's aqi_lag_1 must equal row (i-1)'s aqi
assert sample["aqi_lag_1"].iloc[1] == sample[config.AQI_COL].iloc[0], \
    "aqi_lag_1 is not yesterday's aqi"
# row i's aqi_lag_7 must equal row (i-7)'s aqi
assert sample["aqi_lag_7"].iloc[7] == sample[config.AQI_COL].iloc[0], \
    "aqi_lag_7 is not last week's aqi"

# every city's FIRST row has no yesterday -> all lags NaN at the head
first_rows = df.groupby(config.CITY_ID_COL).head(1)
assert first_rows[lag_cols].isna().all().all(), \
    "some city's first row has a non-NaN lag"

print("\nfirst-row lags all NaN:", bool(first_rows[lag_cols].isna().all().all()))
for c in lag_cols:
    print(f"  {c:>10} NaNs: {df[c].isna().sum()}")
print("shape:", df.shape)

checking city: Agartala, Tripura
      date        aqi  aqi_lag_1  aqi_lag_2  aqi_lag_3  aqi_lag_7
2020-11-08  88.200575        NaN        NaN        NaN        NaN
2020-11-09  70.889445  88.200575        NaN        NaN        NaN
2020-11-10  68.207026  70.889445  88.200575        NaN        NaN
2020-11-11  82.092471  68.207026  70.889445  88.200575        NaN
2020-11-12  75.941422  82.092471  68.207026  70.889445        NaN
2020-11-13  58.662184  75.941422  82.092471  68.207026        NaN
2020-11-14 329.173381  58.662184  75.941422  82.092471        NaN
2020-11-15 305.461395 329.173381  58.662184  75.941422  88.200575
2020-11-16 213.650948 305.461395 329.173381  58.662184  70.889445

first-row lags all NaN: True
   aqi_lag_1 NaNs: 9973
   aqi_lag_2 NaNs: 10113
   aqi_lag_3 NaNs: 10253
   aqi_lag_7 NaNs: 10809
shape: (224808, 21)


## 3. Rolling features: recent level and volatility

A lag is one noisy snapshot. Rolling features summarise a window of recent days.
I build two kinds. Rolling means (3 and 7 day) give the recent baseline level the
city has been sitting at, smoothing out single-day blips. Rolling standard
deviations (3 and 7 day) give recent volatility — how much AQI has been bouncing
around lately.

The volatility features matter most. On the calm days before a festival or dust
event, recent volatility is low; as conditions destabilise it rises before the
peak. That gives the model a warning signal that lags alone can't provide on
regime-change days like Diwali.

Each window ends on day T and includes today, which is legal because T is strictly
before the day I'm forecasting (T+1). I group by city_id so windows never cross a
city boundary, and require the full window (min_periods = window) so a "3-day
average" honestly uses three real days. This is only correct because the grid is
gap-free.

In [5]:
# ROLL_WINDOWS promoted to config.py when FeatureBuilder is assembled
ROLL_WINDOWS = [3, 7]

g = df.groupby(config.CITY_ID_COL)

# For each window: rolling mean (recent level) and rolling std (recent volatility).
# .rolling looks BACKWARD, ending on today (T) -> all data is <= T, so leakage-safe.
# min_periods = window means the first (window-1) rows per city are NaN (warm-up).
# reset_index(level=0, drop=True) strips the city level off the result's MultiIndex
# so it realigns to df's original row order.
roll_cols = []
for w in ROLL_WINDOWS:
    roll = g[config.AQI_COL].rolling(window=w, min_periods=w)

    mean_col = f"aqi_roll_mean_{w}"
    std_col  = f"aqi_roll_std_{w}"

    df[mean_col] = roll.mean().reset_index(level=0, drop=True)
    df[std_col]  = roll.std().reset_index(level=0, drop=True)
    roll_cols += [mean_col, std_col]

# --- prove the rolling math on one city, by hand ---
sample = df[df[config.CITY_ID_COL] == df[config.CITY_ID_COL].iloc[0]]
print("checking city:", sample[config.CITY_ID_COL].iloc[0])
print(
    sample[[config.DATE_COL, config.AQI_COL] + roll_cols]
    .head(8)
    .to_string(index=False)
)

# roll_mean_3 at the 3rd row must equal the plain mean of the first 3 aqi values
manual_mean = sample[config.AQI_COL].iloc[0:3].mean()
assert np.isclose(sample["aqi_roll_mean_3"].iloc[2], manual_mean), \
    "roll_mean_3 does not match a hand-computed 3-day mean"

# roll_std_3 at the 3rd row must equal the plain std (ddof=1) of the first 3 values
manual_std = sample[config.AQI_COL].iloc[0:3].std()
assert np.isclose(sample["aqi_roll_std_3"].iloc[2], manual_std), \
    "roll_std_3 does not match a hand-computed 3-day std"

# warm-up: each city's first (w-1) rows are NaN -> first row is NaN for every roll col
first_rows = df.groupby(config.CITY_ID_COL).head(1)
assert first_rows[roll_cols].isna().all().all(), \
    "some city's first row has a non-NaN rolling value"

print("\nrolling math matches by hand: True")
for c in roll_cols:
    print(f"  {c:>16} NaNs: {df[c].isna().sum()}")
print("shape:", df.shape)

checking city: Agartala, Tripura
      date        aqi  aqi_roll_mean_3  aqi_roll_std_3  aqi_roll_mean_7  aqi_roll_std_7
2020-11-08  88.200575              NaN             NaN              NaN             NaN
2020-11-09  70.889445              NaN             NaN              NaN             NaN
2020-11-10  68.207026        75.765682       10.852132              NaN             NaN
2020-11-11  82.092471        73.729647        7.365559              NaN             NaN
2020-11-12  75.941422        75.413640        6.957752              NaN             NaN
2020-11-13  58.662184        72.232026       12.147605              NaN             NaN
2020-11-14 329.173381       154.592329      151.438274       110.452358       96.920440
2020-11-15 305.461395       231.098987      149.804549       141.489618      120.520837

rolling math matches by hand: True
   aqi_roll_mean_3 NaNs: 13714
    aqi_roll_std_3 NaNs: 13714
   aqi_roll_mean_7 NaNs: 20209
    aqi_roll_std_7 NaNs: 20209
shape: (224808,

## 4. Calendar features: cyclical signals from the date

Lags and rollings capture recent behaviour. Calendar features capture cyclical
behaviour — patterns that repeat by position in the year or week. India's AQI has
a strong seasonal signature: winter is far worse than the monsoon, which scrubs
the air. A model that knows the month carries a seasonal prior before it reads a
single lag.

I extract month, day_of_year, and day_of_week as raw cyclical signals, is_weekend
as a blunt high-value flag for reduced weekday activity, and a human-readable
season bucket mapped from month — partly for the model, but mainly so the Streamlit
app and my seasonal charts have a clean, readable grouping axis later.

These features describe day T itself, which is fully known when forecasting T+1.
No shift, no window, no warm-up NaNs — zero leakage risk by construction.

In [6]:
# MONTH_TO_SEASON promoted to config.py when FeatureBuilder is assembled.
# India-appropriate buckets (not the four temperate seasons).
MONTH_TO_SEASON = {
    12: "Winter",      1: "Winter",      2: "Winter",
     3: "Summer",      4: "Summer",      5: "Summer",
     6: "Monsoon",     7: "Monsoon",     8: "Monsoon",     9: "Monsoon",
    10: "Post-Monsoon", 11: "Post-Monsoon",
}

d = df[config.DATE_COL].dt   # the .dt accessor exposes datetime parts

df["month"]       = d.month
df["day_of_year"] = d.dayofyear
df["day_of_week"] = d.dayofweek           # Monday=0 ... Sunday=6
df["is_weekend"]  = (d.dayofweek >= 5).astype("int8")   # Sat=5, Sun=6
df["season"]      = df["month"].map(MONTH_TO_SEASON)

calendar_cols = ["month", "day_of_year", "day_of_week", "is_weekend", "season"]

# --- eyeball on one city ---
sample = df[df[config.CITY_ID_COL] == df[config.CITY_ID_COL].iloc[0]]
print("checking city:", sample[config.CITY_ID_COL].iloc[0])
print(
    sample[[config.DATE_COL] + calendar_cols]
    .head(8)
    .to_string(index=False)
)

# --- validate the calendar logic, no magic trust ---
# every season maps to a real month bucket (no NaN season anywhere)
assert df["season"].notna().all(), "some rows have no season mapping"
# is_weekend must be exactly the Sat/Sun rows
assert (df["is_weekend"] == (df["day_of_week"] >= 5).astype("int8")).all(), \
    "is_weekend does not match day_of_week"
# ranges are sane
assert df["month"].between(1, 12).all(),        "month out of range"
assert df["day_of_year"].between(1, 366).all(), "day_of_year out of range"
assert df["day_of_week"].between(0, 6).all(),   "day_of_week out of range"
# calendar features NEVER introduce NaNs (the whole point of this bucket)
assert df[calendar_cols].notna().all().all(),   "a calendar feature has NaNs"

print("\nall calendar asserts passed")
print("season distribution:")
print(df["season"].value_counts().to_string())
print("\nweekend share:", round(df["is_weekend"].mean(), 3), "(expect ~0.286)")
print("shape:", df.shape)

checking city: Agartala, Tripura
      date  month  day_of_year  day_of_week  is_weekend       season
2020-11-08     11          313            6           1 Post-Monsoon
2020-11-09     11          314            0           0 Post-Monsoon
2020-11-10     11          315            1           0 Post-Monsoon
2020-11-11     11          316            2           0 Post-Monsoon
2020-11-12     11          317            3           0 Post-Monsoon
2020-11-13     11          318            4           0 Post-Monsoon
2020-11-14     11          319            5           1 Post-Monsoon
2020-11-15     11          320            6           1 Post-Monsoon

all calendar asserts passed
season distribution:
season
Monsoon         71759
Winter          59489
Summer          56892
Post-Monsoon    36668

weekend share: 0.285 (expect ~0.286)
shape: (224808, 30)


## 5. Assemble, drop warm-up rows, audit, save

Now the table becomes model-ready. I declare the feature list, the two targets,
and the metadata columns explicitly. Then I drop — in one honest pass, only now
that every feature exists — any row missing a target or any feature. Finally I
run a mechanical leakage audit and save features.parquet for the modeling sessions.

In [7]:
# --- column contract (all promoted to config.py when FeatureBuilder is built) ---
LAG_COLS   = [f"aqi_lag_{k}" for k in [1, 2, 3, 7]]
ROLL_COLS  = [f"aqi_roll_{stat}_{w}"
              for w in [3, 7] for stat in ("mean", "std")]
CAL_NUMERIC_COLS = ["month", "day_of_year", "day_of_week", "is_weekend"]

# today's pollutant LEVELS are legal day-T features
POLLUTANT_FEATURES = config.MODEL_POLLUTANT_COLS   # the 9 kept pollutants

FEATURE_COLS = POLLUTANT_FEATURES + LAG_COLS + ROLL_COLS + CAL_NUMERIC_COLS

TARGET_REG = "target_aqi"
TARGET_CLF = "target_bucket"

# kept for the app + time-aware splitting, never fed to the model as features
META_COLS = [config.CITY_ID_COL, config.CITY_COL, config.STATE_COL,
             config.DATE_COL, config.AQI_COL, config.AQI_BUCKET_COL, "season"]

print("n features :", len(FEATURE_COLS))
print("features   :", FEATURE_COLS)
print("targets    :", TARGET_REG, "|", TARGET_CLF)

n features : 21
features   : ['pm25', 'pm10', 'no', 'no2', 'nox', 'nh3', 'so2', 'co', 'o3', 'aqi_lag_1', 'aqi_lag_2', 'aqi_lag_3', 'aqi_lag_7', 'aqi_roll_mean_3', 'aqi_roll_std_3', 'aqi_roll_mean_7', 'aqi_roll_std_7', 'month', 'day_of_year', 'day_of_week', 'is_weekend']
targets    : target_aqi | target_bucket


In [11]:
# how much does EACH required column contribute to the drop?
required = FEATURE_COLS + [TARGET_REG, TARGET_CLF]

print("NaN count per required column (on the assembled, pre-drop frame):")
nan_counts = model_df_predrop = df[META_COLS + required].copy()
for c in required:
    n = model_df_predrop[c].isna().sum()
    pct = n / len(model_df_predrop) * 100
    print(f"  {c:>16}: {n:>7}  ({pct:4.1f}%)")

# split the blame: structural (lag/roll/target) vs pollutant levels
structural = LAG_COLS + ROLL_COLS + [TARGET_REG, TARGET_CLF]
pollutant  = POLLUTANT_FEATURES

drop_structural = model_df_predrop[structural].isna().any(axis=1).sum()
drop_pollutant  = model_df_predrop[pollutant].isna().any(axis=1).sum()
drop_either     = model_df_predrop[required].isna().any(axis=1).sum()

print("\nrows lost if we drop on STRUCTURAL only:", drop_structural,
      f"({drop_structural/len(model_df_predrop):.1%})")
print("rows lost if we drop on POLLUTANTS only :", drop_pollutant,
      f"({drop_pollutant/len(model_df_predrop):.1%})")
print("rows lost if we drop on EITHER (current):", drop_either,
      f"({drop_either/len(model_df_predrop):.1%})")

# which cities die ONLY because of pollutant NaNs?
survives_structural = ~model_df_predrop[structural].isna().any(axis=1)
cities_struct = df.loc[survives_structural, config.CITY_ID_COL].nunique()
print("\ncities surviving structural-only drop:", cities_struct, "(vs 113 now)")

# worst pollutant offenders by % missing
print("\npollutant missingness ranked:")
for c in sorted(pollutant, key=lambda x: -model_df_predrop[x].isna().mean()):
    print(f"  {c:>6}: {model_df_predrop[c].isna().mean():.1%}")

NaN count per required column (on the assembled, pre-drop frame):
              pm25:   13539  ( 6.0%)
              pm10:   19695  ( 8.8%)
                no:   23784  (10.6%)
               no2:   11723  ( 5.2%)
               nox:   26502  (11.8%)
               nh3:   43554  (19.4%)
               so2:   12467  ( 5.5%)
                co:   20290  ( 9.0%)
                o3:   23758  (10.6%)
         aqi_lag_1:    9973  ( 4.4%)
         aqi_lag_2:   10113  ( 4.5%)
         aqi_lag_3:   10253  ( 4.6%)
         aqi_lag_7:   10809  ( 4.8%)
   aqi_roll_mean_3:   13714  ( 6.1%)
    aqi_roll_std_3:   13714  ( 6.1%)
   aqi_roll_mean_7:   20209  ( 9.0%)
    aqi_roll_std_7:   20209  ( 9.0%)
             month:       0  ( 0.0%)
       day_of_year:       0  ( 0.0%)
       day_of_week:       0  ( 0.0%)
        is_weekend:       0  ( 0.0%)
        target_aqi:    9973  ( 4.4%)
     target_bucket:    9973  ( 4.4%)

rows lost if we drop on STRUCTURAL only: 23083 (10.3%)
rows lost if we drop on POL

## 5b. Drop structurally-unusable rows only

I split NaNs by cause. Structural NaNs — lags, rollings, and the targets — are
unfixable: I can't invent a city's first yesterday or learn from a missing label,
so those rows are dropped. Pollutant NaNs are ordinary missingness; deleting them
here would throw away ~26% of rows and 28 whole cities for data an imputer can
fill safely inside the CV fold. So I leave pollutant gaps in and hand them to the
Pipeline's imputer in Session 5.

In [12]:
# assemble: meta + features + both targets
keep = META_COLS + FEATURE_COLS + [TARGET_REG, TARGET_CLF]
model_df = df[keep].copy()

# STRUCTURAL columns: NaNs here are unfixable (no history / no label) -> drop.
# Pollutant levels are deliberately EXCLUDED: ordinary missingness, imputed later.
STRUCTURAL_COLS = LAG_COLS + ROLL_COLS + [TARGET_REG, TARGET_CLF]

rows_before = len(model_df)
model_df = model_df.dropna(subset=STRUCTURAL_COLS).reset_index(drop=True)
rows_after = len(model_df)

print("rows before drop :", rows_before)
print("rows after drop  :", rows_after)
print("rows dropped     :", rows_before - rows_after,
      f"({(rows_before - rows_after) / rows_before:.1%})")
print("cities remaining :", model_df[config.CITY_ID_COL].nunique())
print("date range       :", model_df[config.DATE_COL].min(),
      "->", model_df[config.DATE_COL].max())

# --- hard floors ---
# 1) zero structural NaNs survive
assert model_df[STRUCTURAL_COLS].notna().all().all(), "structural NaNs survived"
# 2) all 141 cities must survive
assert model_df[config.CITY_ID_COL].nunique() == 141, \
    f"expected 141 cities, got {model_df[config.CITY_ID_COL].nunique()}"
# 3) pollutant NaNs SHOULD still be present — surfaces the Session-5 imputation duty
pollutant_nans = model_df[POLLUTANT_FEATURES].isna().sum().sum()
assert pollutant_nans > 0, "expected pollutant NaNs to remain for downstream imputation"
print(f"\npollutant NaNs remaining: {pollutant_nans} "
      f"(MUST be imputed in the Session 5 Pipeline, train-fold only)")

rows before drop : 224808
rows after drop  : 201725
rows dropped     : 23083 (10.3%)
cities remaining : 141
date range       : 2018-01-08 00:00:00 -> 2023-03-30 00:00:00

pollutant NaNs remaining: 117950 (MUST be imputed in the Session 5 Pipeline, train-fold only)


In [13]:
# brute-force leakage audit on one busy city: reconstruct each feature from
# the ORIGINAL gap-free frame using only data at day T or earlier, and confirm
# the target is genuinely day T+1.
audit_city = model_df[config.CITY_ID_COL].value_counts().index[0]
orig = (df[df[config.CITY_ID_COL] == audit_city]
        .set_index(config.DATE_COL).sort_index())

checks = 0
for _, row in model_df[model_df[config.CITY_ID_COL] == audit_city].head(200).iterrows():
    t = row[config.DATE_COL]

    # target must equal the ORIGINAL aqi exactly one day later
    t_next = t + pd.Timedelta(days=1)
    assert np.isclose(row[TARGET_REG], orig.loc[t_next, config.AQI_COL]), \
        f"target_aqi at {t} is not next-day aqi"

    # lag_1 must equal the ORIGINAL aqi exactly one day earlier
    t_prev = t - pd.Timedelta(days=1)
    assert np.isclose(row["aqi_lag_1"], orig.loc[t_prev, config.AQI_COL]), \
        f"aqi_lag_1 at {t} is not previous-day aqi"

    # roll_mean_3 must equal mean of ORIGINAL aqi over [T-2, T-1, T]
    window = orig.loc[t - pd.Timedelta(days=2): t, config.AQI_COL]
    assert np.isclose(row["aqi_roll_mean_3"], window.mean()), \
        f"roll_mean_3 at {t} is not the trailing 3-day mean"

    checks += 1

print(f"leakage audit passed on '{audit_city}': {checks} rows verified")
print("  - every target is genuinely day T+1")
print("  - every feature is reconstructable from day T or earlier")

leakage audit passed on 'Noida, Uttar Pradesh': 200 rows verified
  - every target is genuinely day T+1
  - every feature is reconstructable from day T or earlier


In [14]:
# FEATURES_PATH promoted to config.py when FeatureBuilder is assembled
FEATURES_PATH = config.PROCESSED_DIR / "features.parquet"

model_df.to_parquet(FEATURES_PATH, index=False)
print("saved:", FEATURES_PATH)
print("shape:", model_df.shape)

# round-trip check: reload and confirm identical shape
reloaded = pd.read_parquet(FEATURES_PATH)
assert reloaded.shape == model_df.shape, "round-trip shape mismatch"
print("round-trip OK")

saved: D:\07_Data_Science\05_DS_Projects\03_AirCast\data\processed\features.parquet
shape: (201725, 30)
round-trip OK


In [15]:
from src.features.builder import FeatureBuilder

clean = pd.read_parquet(config.CLEAN_CITY_DAY_PATH)
built = FeatureBuilder().build(clean)
saved = pd.read_parquet(config.FEATURES_PATH)   # the table we built by hand

print("built shape:", built.shape)
print("saved shape:", saved.shape)
assert list(built.columns) == list(saved.columns), "column mismatch"

# value-for-value equality with the hand-built table
pd.testing.assert_frame_equal(
    built.reset_index(drop=True),
    saved.reset_index(drop=True),
    check_dtype=False,
)
print("FeatureBuilder reproduces the notebook table exactly ✅")

built shape: (201725, 30)
saved shape: (201725, 30)
FeatureBuilder reproduces the notebook table exactly ✅
